# 📓 Notebook 2 — ETL & Preprocessing
### AirSense AI — Intelligent Urban Air Quality Forecasting & Decision Support System

**Purpose of this notebook**

Turn the two cleaned-but-separate tables from Notebook 1 into a single,
analysis-ready table: the **AirSense Feature Cube**.

Steps performed here, in order:

1. Load the parsed checkpoints from Notebook 1
2. Localize timestamps: GMT → `America/New_York`
3. Rename columns to friendly names (`pm2_5` → `PM2.5`, etc.)
4. Merge Air Quality + Weather on `time`
5. Re-check for missing values / duplicates post-merge
6. Detect outliers
7. Generate calendar features (Hour, Weekday, Month, Season, Weekend, ISO Week)
8. Save the final `airsense_cube.csv`

---


## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

print("pandas version:", pd.__version__)

pandas version: 2.3.3


## 2. Load the parsed checkpoints from Notebook 1

These are the datetime-converted, artifact-cleaned versions saved at the end
of Notebook 1 — not the raw `.xlsx` files.

In [2]:
PROJECT_ROOT = Path(r"C:\Users\pc\Desktop\AirSenseAI")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

AIR_QUALITY_PARSED = RAW_DATA_DIR / "air_quality_parsed.csv"
WEATHER_PARSED = RAW_DATA_DIR / "weather_parsed.csv"

print("Air quality checkpoint:", AIR_QUALITY_PARSED, "| exists:", AIR_QUALITY_PARSED.exists())
print("Weather checkpoint:    ", WEATHER_PARSED, "| exists:", WEATHER_PARSED.exists())

Air quality checkpoint: C:\Users\pc\Desktop\AirSenseAI\data\raw\air_quality_parsed.csv | exists: True
Weather checkpoint:     C:\Users\pc\Desktop\AirSenseAI\data\raw\weather_parsed.csv | exists: True


In [3]:
df_air = pd.read_csv(AIR_QUALITY_PARSED, parse_dates=['time'])
df_weather = pd.read_csv(WEATHER_PARSED, parse_dates=['time'])

print("Air quality shape:", df_air.shape)
print("Weather shape:    ", df_weather.shape)

df_air.head()

Air quality shape: (17568, 5)
Weather shape:     (17568, 7)


,time,pm10 (μg/m³),pm2_5 (μg/m³),ozone (μg/m³),nitrogen_dioxide (μg/m³)
0,2024-01-01 00:00:00,27.8,19.3,0,56.8
1,2024-01-01 01:00:00,31.3,21.7,0,55.6
2,2024-01-01 02:00:00,35.5,24.6,0,54.0
3,2024-01-01 03:00:00,40.0,27.8,0,52.5
4,2024-01-01 04:00:00,43.9,30.5,0,51.5


## 3. Localize timestamps: GMT → America/New_York

Notebook 1 confirmed the raw timestamps are GMT-naive (no timezone attached,
but representing GMT time). We now:

1. **Localize** them as GMT (attach the GMT label, values don't change)
2. **Convert** to `America/New_York` (values shift to account for the
   UTC-4/UTC-5 offset, and DST transitions are handled automatically by pandas)

This matters because PM2.5 forecasting is inherently about *local* daily and
weekly cycles (rush hour, weekday/weekend patterns) — those only make sense
in NYC local time, not GMT.

In [4]:
df_air['time'] = df_air['time'].dt.tz_localize('GMT').dt.tz_convert('America/New_York')
df_weather['time'] = df_weather['time'].dt.tz_localize('GMT').dt.tz_convert('America/New_York')

print("Air quality time range (local):", df_air['time'].min(), "→", df_air['time'].max())
print("Weather time range (local):    ", df_weather['time'].min(), "→", df_weather['time'].max())

Air quality time range (local): 2023-12-31 19:00:00-05:00 → 2026-01-01 18:00:00-05:00
Weather time range (local):     2023-12-31 19:00:00-05:00 → 2026-01-01 18:00:00-05:00


**Note on DST:** converting to `America/New_York` means the clock jumps
forward/back on daylight saving transition days. This can cause one
"missing" hour in spring and one "duplicate" local hour in fall, purely as an
artifact of the timezone conversion — not a real data problem. We check for
this explicitly in Section 6.

## 4. Rename columns to friendly names

Raw Open-Meteo names (with units embedded) get renamed to clean, consistent
names that the rest of the project (EDA, stats, modeling, dashboard) will use
going forward.

In [5]:
air_rename_map = {
    "pm10 (μg/m³)": "PM10",
    "pm2_5 (μg/m³)": "PM2.5",
    "ozone (μg/m³)": "O3",
    "nitrogen_dioxide (μg/m³)": "NO2",
}

weather_rename_map = {
    "temperature_2m (°C)": "Temperature",
    "relative_humidity_2m (%)": "Relative Humidity",
    "wind_speed_10m (km/h)": "Wind Speed",
    "wind_direction_10m (°)": "Wind Direction",
    "precipitation (mm)": "Precipitation",
    "cloud_cover (%)": "Cloud Cover",
}

df_air = df_air.rename(columns=air_rename_map)
df_weather = df_weather.rename(columns=weather_rename_map)

print("Air quality columns:", list(df_air.columns))
print("Weather columns:    ", list(df_weather.columns))

Air quality columns: ['time', 'PM10', 'PM2.5', 'O3', 'NO2']
Weather columns:     ['time', 'Temperature', 'Relative Humidity', 'Wind Speed', 'Wind Direction', 'Precipitation', 'Cloud Cover']


⚠️ **Reminder:** there is no `Surface Pressure` column in this dataset —
the original project plan expected one, but it isn't present in the raw
files. Every notebook from here on works without it.

## 5. Merge Air Quality + Weather on `time`

An **inner join** on `time` keeps only timestamps present in both tables.
Since Notebook 1 confirmed both files cover the identical range with zero
gaps, this should not drop any rows — but we verify that assumption instead
of just trusting it.

In [6]:
df_cube = pd.merge(df_air, df_weather, on='time', how='inner')

print("Air quality rows:", df_air.shape[0])
print("Weather rows:    ", df_weather.shape[0])
print("Merged rows:     ", df_cube.shape[0])

if df_cube.shape[0] < min(df_air.shape[0], df_weather.shape[0]):
    print("⚠️ Merge dropped rows — investigate mismatched timestamps.")
else:
    print("✅ Merge kept all matching rows, as expected.")

df_cube.head()

Air quality rows: 17568
Weather rows:     17568
Merged rows:      17568
✅ Merge kept all matching rows, as expected.


,time,PM10,PM2.5,O3,NO2,Temperature,Relative Humidity,Wind Speed,Wind Direction,Precipitation,Cloud Cover
0,2023-12-31 19:00:00-05:00,27.8,19.3,0,56.8,3.8,65,5.7,235,0.0,65
1,2023-12-31 20:00:00-05:00,31.3,21.7,0,55.6,3.8,60,6.0,245,0.0,99
2,2023-12-31 21:00:00-05:00,35.5,24.6,0,54.0,3.1,65,6.8,238,0.0,96
3,2023-12-31 22:00:00-05:00,40.0,27.8,0,52.5,2.6,68,8.4,245,0.0,73
4,2023-12-31 23:00:00-05:00,43.9,30.5,0,51.5,1.9,74,6.3,246,0.0,100


## 6. Re-check missing values, duplicates, and DST artifacts post-merge

Even though Notebook 1 found the raw data clean, the timezone conversion in
Section 3 can introduce its own quirks (DST transitions), so we check again
here on the merged table.

In [7]:
print("=== Missing values (post-merge) ===")
print(df_cube.isna().sum())

=== Missing values (post-merge) ===
time                 0
PM10                 0
PM2.5                0
O3                   0
NO2                  0
Temperature          0
Relative Humidity    0
Wind Speed           0
Wind Direction       0
Precipitation        0
Cloud Cover          0
dtype: int64


In [8]:
dupe_count = df_cube['time'].duplicated().sum()
print("Duplicate timestamps post-merge:", dupe_count)

if dupe_count > 0:
    print("Likely cause: fall-back DST transition. Showing duplicates:")
    display(df_cube[df_cube['time'].duplicated(keep=False)].sort_values('time'))

Duplicate timestamps post-merge: 0


In [9]:
full_range = pd.date_range(df_cube['time'].min(), df_cube['time'].max(), freq='h', tz='America/New_York')
missing_hours = full_range.difference(df_cube['time'])
print("Missing local hours post-conversion:", len(missing_hours))

if len(missing_hours) > 0:
    print("Likely cause: spring-forward DST transition. Missing hours:")
    print(missing_hours)

Missing local hours post-conversion: 0


**Handling duplicates/gaps, if any were found above:**
- A duplicate hour from "fall back" is usually resolved by averaging the two
  readings, or keeping the first occurrence — pick whichever is simpler for
  your use case and document the choice.
- A missing hour from "spring forward" simply doesn't exist in local time and
  can be left as a gap, or interpolated from neighboring hours.

If both counts above came back as 0, no action is needed — proceed to the
next section.

## 7. Detect outliers

A simple, transparent approach: flag values outside
**mean ± 3 standard deviations** for each numeric column. This doesn't remove
anything automatically — it just surfaces candidates for manual review.

In [10]:
numeric_cols = ['PM10', 'PM2.5', 'O3', 'NO2', 'Temperature',
                'Relative Humidity', 'Wind Speed', 'Wind Direction',
                'Precipitation', 'Cloud Cover']

outlier_summary = {}
for col in numeric_cols:
    mean, std = df_cube[col].mean(), df_cube[col].std()
    lower, upper = mean - 3 * std, mean + 3 * std
    outliers = df_cube[(df_cube[col] < lower) | (df_cube[col] > upper)]
    outlier_summary[col] = len(outliers)

pd.Series(outlier_summary, name="outlier_count").sort_values(ascending=False)

PM2.5                278
PM10                 275
O3                   264
Precipitation        264
Wind Speed           160
NO2                  100
Temperature            0
Relative Humidity      0
Wind Direction         0
Cloud Cover            0
Name: outlier_count, dtype: int64

**Observation:** _(fill this in after running the cell above)_
Note which columns show the most outliers. For a pollutant like PM2.5, a few
extreme spikes (e.g. wildfire smoke events, fireworks on holidays) are
often *real* and meaningful — not necessarily data errors. Cross-check any
large outlier count against known events before deciding whether to cap,
remove, or keep these values. That deeper judgment call happens in
Notebook 3 (EDA), where we can visualize these points in context.

## 8. Generate calendar features

These features let later models (and the dashboard) pick up on daily,
weekly, and seasonal patterns in air quality — e.g. rush-hour PM2.5 spikes,
weekday vs weekend traffic differences, winter heating season effects.

In [11]:
df_cube['Hour'] = df_cube['time'].dt.hour
df_cube['Weekday'] = df_cube['time'].dt.day_name()
df_cube['Month'] = df_cube['time'].dt.month
df_cube['Weekend'] = df_cube['time'].dt.dayofweek >= 5
df_cube['ISO Week'] = df_cube['time'].dt.isocalendar().week

def month_to_season(month):
    if month in (12, 1, 2):
        return "Winter"
    elif month in (3, 4, 5):
        return "Spring"
    elif month in (6, 7, 8):
        return "Summer"
    else:
        return "Fall"

df_cube['Season'] = df_cube['Month'].apply(month_to_season)

df_cube[['time', 'Hour', 'Weekday', 'Month', 'Season', 'Weekend', 'ISO Week']].head()

,time,Hour,Weekday,Month,Season,Weekend,ISO Week
0,2023-12-31 19:00:00-05:00,19,Sunday,12,Winter,True,52
1,2023-12-31 20:00:00-05:00,20,Sunday,12,Winter,True,52
2,2023-12-31 21:00:00-05:00,21,Sunday,12,Winter,True,52
3,2023-12-31 22:00:00-05:00,22,Sunday,12,Winter,True,52
4,2023-12-31 23:00:00-05:00,23,Sunday,12,Winter,True,52


## 9. Final structure check

One last look at the complete feature cube before saving — shape, dtypes,
and a sample of rows.

In [12]:
print("Final shape:", df_cube.shape)
print()
df_cube.info()

Final shape: (17568, 17)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17568 entries, 0 to 17567
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype                           
---  ------             --------------  -----                           
 0   time               17568 non-null  datetime64[ns, America/New_York]
 1   PM10               17568 non-null  float64                         
 2   PM2.5              17568 non-null  float64                         
 3   O3                 17568 non-null  int64                           
 4   NO2                17568 non-null  float64                         
 5   Temperature        17568 non-null  float64                         
 6   Relative Humidity  17568 non-null  int64                           
 7   Wind Speed         17568 non-null  float64                         
 8   Wind Direction     17568 non-null  int64                           
 9   Precipitation      17568 non-null  float64               

In [ ]:
df_cube.sample(5, random_state=42).sort_values('time')

## 10. Save the AirSense Feature Cube

This is the single processed table that every notebook from here on
(EDA, Statistical Analysis, Feature Engineering, Modeling) will load instead
of touching the raw files again.

In [13]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
output_path = PROCESSED_DATA_DIR / "airsense_cube.csv"

df_cube.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Final shape:", df_cube.shape)

Saved: C:\Users\pc\Desktop\AirSenseAI\data\processed\airsense_cube.csv
Final shape: (17568, 17)


## 11. Summary

At this point we have:

- ✅ Converted timestamps from GMT to `America/New_York`, checked for DST
  artifacts (duplicate/missing local hours)
- ✅ Renamed all columns to clean, consistent names
- ✅ Merged Air Quality + Weather into one table on `time`
- ✅ Re-verified missing values and duplicates post-merge
- ✅ Flagged outlier candidates per column (mean ± 3 std) for later review
- ✅ Generated calendar features: Hour, Weekday, Month, Season, Weekend, ISO Week
- ✅ Saved `data/processed/airsense_cube.csv` — the single source of truth
  for every notebook from here on

**Next step → Notebook 3 (Exploratory Data Analysis):**
Visualize trends in PM2.5, PM10, NO2, O3, and all weather variables; look at
daily/weekly/seasonal cycles; build a correlation heatmap; inspect the
outlier candidates flagged above in visual context.